In [1]:
# 1. Download the raw zipped file directly
!kaggle datasets download -d mbkinaci/fruit-images-for-object-detection

# 2. Quietly unzip it directly into your working directory
!unzip -q fruit-images-for-object-detection.zip

# 3. Clean up the zip file to save disk space
!rm fruit-images-for-object-detection.zip


Dataset URL: https://www.kaggle.com/datasets/mbkinaci/fruit-images-for-object-detection
License(s): CC0-1.0
100%|███████████████████████████████████████| 28.4M/28.4M [00:00<00:00, 114MB/s]



In [2]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Flatten,Dropout,BatchNormalize
from keras.applications.vgg16 import VGG16

ImportError: cannot import name 'BatchNormalize' from 'keras.layers' (/usr/local/lib/python3.13/dist-packages/keras/layers/__init__.py)

In [ ]:
conv_base=VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=((150,150,3)),
)

In [ ]:
model=Sequential()
model.add(conv_base)
model.add(Flatten())
model.add(Dense(256,activation='relu'))
model.add(BatchNormalize())
model.add(Dropout(0.5))
model.add(Dense(4,activation='softmax'))

In [ ]:
model.summary()

In [ ]:
conv_base.trainable=False

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    "/kaggle/input/datasets/mbkinaci/fruit-images-for-object-detection",
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(150, 150),
    batch_size=12
)

In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    "/kaggle/input/datasets/mbkinaci/fruit-images-for-object-detection",
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(150, 150),
    batch_size=12
)

In [ ]:
#normalize
def normalize(img,label):
    img=tf.cast(img/255,tf.float32)
    return img,label
train_ds=train_ds.map(normalize)
val_ds=val_ds.map(normalize)

In [ ]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [ ]:
history=model.fit(train_ds, validation_data=val_ds,epochs=10)